## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 02: Data Cleaning and Exploratory Analysis**

[![Pandas](https://img.shields.io/badge/Pandas-Latest-green)](https://pandas.pydata.org/) [![Matplotlib](https://img.shields.io/badge/Matplotlib-Latest-orange)](https://matplotlib.org/)

---

**Scope:** Run data cleaning checks (nulls, duplicate timestamps, price anomalies) and perform exploratory data analysis of price history, log returns, and volumes.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 2. **02_data_cleaning_eda.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
This notebook cleans the raw market dataset, verifies its integrity, and performs baseline descriptive analysis. We check that daily close prices are positive and remove duplicate date entries. The cleaned dataset is then saved in Parquet format to `data/interim/` to serve as the input for feature engineering.

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
print("[OK] Setup complete.")

[OK] Setup complete.


## 3. Data Loading and Schema Audit

In [2]:
df_raw = pd.read_csv(config.RAW_DATA_DIR / "banknifty_ohlcv_2023_2026.csv")
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
print(df_raw.info())
df_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 755 entries, 0 to 754
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    755 non-null    datetime64[ns]
 1   Close   755 non-null    float64       
 2   High    755 non-null    float64       
 3   Low     755 non-null    float64       
 4   Open    755 non-null    float64       
 5   Volume  755 non-null    int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 35.5 KB
None


,Date,Close,High,Low,Open,Volume
0,2023-06-01,43790.199219,44179.949219,43706.148438,44172.050781,182900
1,2023-06-02,43937.851562,44088.898438,43812.550781,43997.300781,123900
2,2023-06-05,44101.648438,44266.148438,44074.199219,44103.250000,124500
3,2023-06-06,44164.550781,44236.199219,44009.699219,44156.699219,123600
4,2023-06-07,44275.300781,44346.300781,44143.500000,44318.699219,119200


## 4. Cleaning and Quality Controls

In [3]:
# Check missing values
print("Missing values summary:")
print(df_raw.isnull().sum())

# Ensure positive Close prices
assert (df_raw['Close'] > 0).all(), "Negative close prices found!"
print("[OK] Close price positivity checked.")

# Remove duplicates
df_clean = df_raw.drop_duplicates(subset=['Date']).sort_values('Date')
print(f"Cleaned observations: {len(df_clean)} (from {len(df_raw)} raw rows)")

Missing values summary:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64
[OK] Close price positivity checked.
Cleaned observations: 755 (from 755 raw rows)


## 5. Exploratory Data Analysis

In [4]:
print("Descriptive Statistics:")
print(df_clean[['Open', 'High', 'Low', 'Close', 'Volume']].describe())

Descriptive Statistics:
               Open          High           Low         Close        Volume
count    755.000000    755.000000    755.000000    755.000000  7.550000e+02
mean   51526.455914  51821.385063  51223.639482  51528.009065  3.864682e+05
std     4931.844014   4956.573707   4927.639189   4950.536067  4.873627e+06
min    42555.601562  42716.199219  42105.398438  42280.148438  0.000000e+00
25%    47547.050781  47847.701172  47258.400391  47518.650391  1.373500e+05
50%    51437.449219  51801.148438  51137.500000  51473.050781  1.800000e+05
75%    55515.150391  55765.449219  55189.925781  55510.250000  2.636000e+05
max    61660.851562  61764.851562  61100.898438  61550.800781  1.340894e+08


## 6. Saving Interim Dataset

In [5]:
df_clean.to_parquet(config.INTERIM_DATA_DIR / "banknifty_clean.parquet", index=False)
print(f"Saved clean interim data to {config.INTERIM_DATA_DIR / 'banknifty_clean.parquet'}")

Saved clean interim data to E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml\data\interim\banknifty_clean.parquet
